
# 13. Batch Eye + OptiTrack Ingest Template (Adapted)

Adapted from `notebooks/tobiasr/data ingest updates/Natasha Batch Ingest Eye Optitrack.ipynb`
and related eye/optitrack ingest workflows.

This template is read-only by default and is intended for batch planning + QC before population.


In [ ]:

import os
from pathlib import Path
import pandas as pd
import datajoint as dj

if Path.cwd().name == "notebooks":
    os.chdir("..")

repo_root = Path.cwd()
cfg = repo_root / "dj_local_conf.json"
if not cfg.exists():
    raise FileNotFoundError("Missing dj_local_conf.json in repository root")

dj.config.load(str(cfg))
dj.conn()

from adamacs.pipeline import subject, session, scan, event, model, behavior
from adamacs.schemas import mocap, virtual_markers_optitrack, pupil_tracking

ALLOW_DB_WRITES = False
INITIALS = os.environ.get("ADAMACS_INITIALS", "NK")
DATE_FROM = os.environ.get("ADAMACS_DATE_FROM", "2025-01-01")


In [ ]:

candidate_scans = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

print("candidate scans:", len(candidate_scans))
pd.DataFrame(candidate_scans).head(20)


In [ ]:

def has_rows(table, key):
    try:
        return len(table & key) > 0
    except Exception:
        return False

rows = []
for key in candidate_scans:
    rows.append(
        {
            **key,
            "has_behavior_recording": has_rows(event.BehaviorRecording, key),
            "has_video_recording": has_rows(model.VideoRecordingNew, key),
            "has_camsync": has_rows(behavior.CamSyncRecording, key),
            "has_optitrack": has_rows(mocap.MotionCapture, key),
            "has_rigidmouse": has_rows(virtual_markers_optitrack.RigidMouseTracking, key),
            "has_pupil_fit": has_rows(pupil_tracking.PupilEllipseFittingFreeMoving, key),
            "has_gaze3d": has_rows(pupil_tracking.GazeReconstruction3D, key),
        }
    )

coverage = pd.DataFrame(rows)
coverage


In [ ]:

missing_any = coverage[
    ~coverage[
        [
            "has_behavior_recording",
            "has_video_recording",
            "has_camsync",
            "has_optitrack",
            "has_rigidmouse",
            "has_pupil_fit",
            "has_gaze3d",
        ]
    ].all(axis=1)
]

missing_any


In [ ]:

# Optional population plan (disabled by default):
if ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES manually after explicit approval.")

# Example execution order for write-enabled runs:
# 1) behavior.CamSyncRecording.populate(...)
# 2) mocap.MotionCapture.populate(...)
# 3) virtual_markers_optitrack.RigidMouseTracking.populate(...)
# 4) pupil_tracking.PupilEllipseFittingFreeMoving.populate(...)
# 5) pupil_tracking.GazeReconstruction3D.populate(...)
